# Unit 2 Exercise


In [ ]:
import re                  
import sys                

try:
    import nltk
    from nltk.corpus import webtext
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk"])
    import nltk
    from nltk.corpus import webtext

try:
    import pdfplumber
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pdfplumber"])
    import pdfplumber

print("All imports successful.")

---

## 1. Updating ELIZA

In [ ]:
# ─────────────────────────────────────────────
# Exercise 1 – Updated ELIZA Implementation
# ─────────────────────────────────────────────

previous_inputs = set()   # Stores normalized past inputs for repetition detection


def normalize(text: str) -> str:
    """
    BONUS – Repeated Question Detection.

    Normalizes text for equality checking:
      1. Lowercases and strips surrounding whitespace.
      2. Removes all non-word characters (punctuation, spaces) so that
         'I feel sad.' and 'i feel sad' map to the same key 'ifeelsad'.
    """
    text = text.lower().strip()
    text = re.sub(r"\W+", "", text)   # \W+ → one-or-more non-word chars
    return text


def reflect(fragment: str) -> str:
    """
    Pronoun reflection maps first-person tokens to second-person and vice
    versa, making ELIZA's responses grammatically coherent when echoing the
    user's captured group back to them (e.g. 'I feel' → 'you feel').
    """
    reflections = {
        "am": "are",
        "was": "were",
        "i": "you",
        "i'd": "you would",
        "i've": "you have",
        "i'll": "you will",
        "my": "your",
        "are": "am",
        "you've": "I have",
        "you'll": "I will",
        "your": "my",
        "yours": "mine",
        "you": "me",
        "me": "you",
    }
    words = fragment.lower().split()
    return " ".join([reflections.get(word, word) for word in words])


def eliza_response(user_input: str) -> str:
    """
    Generates an ELIZA-style therapeutic response.

    Process:
      1. Normalize the input and check against the seen-inputs set (BONUS).
      2. Iterate through the pattern list in order; return on first match.
      3. Fall back to a generic probe if no pattern matches.

    New patterns (a–e) use anchors (^…$) and non-capturing groups (?:…)
    for alternation, ensuring clean matching without spurious group captures.
    """

    # ── BONUS: Repeated-Question Detection ──────────────────────────────────
    norm = normalize(user_input)
    if norm in previous_inputs:
        return "You already asked that. Are you trying to confuse me on purpose?"
    previous_inputs.add(norm)

    # ── Pattern–Response Table ───────────────────────────────────────────────
    patterns = [
        # ── Existing base patterns ──────────────────────────────────────────
        # '(.*)' captures the remainder of the utterance for reflection.
        (r"^I need (.*)$",
         "Why do you need {0}?"),

        # (?:don't|don't) is a non-capturing alternation handling both
        # the straight apostrophe (') and the curly/typographic one (').
        (r"^Why (?:don't|don\u2019t) you (.*)$",
         "Do you really think I don't {0}?"),

        (r"^I feel (.*)$",
         "Tell me more about feeling {0}."),

        # ── NEW Pattern (a) ─────────────────────────────────────────────────
        # Anchors (^ $) enforce a full-line match.
        # \.?$ makes the trailing period optional so the pattern handles
        # input with or without terminal punctuation.
        (r"^I want to know the reasons why I am feeling depressed all the time\.?$",
         "What do you think might be the reasons you feel depressed all the time?"),

        # ── NEW Pattern (b) ─────────────────────────────────────────────────
        # Exact phrase match; optional period at the end.
        (r"^I am feeling stressed\.?$",
         "What do you think is causing you to feel stressed?"),

        # ── NEW Pattern (c) ─────────────────────────────────────────────────
        # Captures the specific phrasing about a crush for targeted response.
        (r"^My feelings towards my crush are invalidated\.?$",
         "How does it feel when your feelings are invalidated?"),

        # ── NEW Pattern (d) ─────────────────────────────────────────────────
        # (?:don't|don\u2019t|do not) handles three surface forms:
        #   • straight apostrophe contraction: don't
        #   • typographic/curly apostrophe:    don't
        #   • full two-word form:              do not
        (r"^You (?:don't|don\u2019t|do not) understand me\.?$",
         "What makes you think I don't understand you?"),

        # ── NEW Pattern (e) ─────────────────────────────────────────────────
        # (?:can't|can\u2019t|cannot) collapses three equivalent surface forms
        # into a single pattern branch.
        (r"^I (?:can't|can\u2019t|cannot) focus on my studies\.?$",
         "What do you think is making it hard for you to focus on your studies?"),
    ]

    for pattern, response_template in patterns:
        match = re.match(pattern, user_input, re.IGNORECASE)
        if match:
            # Only patterns containing (.*) will have a captured group.
            if match.lastindex:
                return response_template.format(reflect(match.group(1)))
            return response_template

    # ── Fallback ─────────────────────────────────────────────────────────────
    return "Can you tell me more?"


# ── Demo: exercise each of the 5 required inputs + the BONUS ────────────────
demo_inputs = [
    # Required patterns a–e
    "I want to know the reasons why I am feeling depressed all the time.",
    "I am feeling stressed.",
    "My feelings towards my crush are invalidated.",
    "You don't understand me.",
    "I can't focus on my studies.",
    # BONUS – exact repeat
    "I am feeling stressed.",
]

print("ELIZA Demo\n" + "=" * 50)
for utt in demo_inputs:
    print(f"You : {utt}")
    print(f"ELIZA: {eliza_response(utt)}")
    print()

---

## 2. Implementing RegEx on NLP


### 2a. Extracting Words Starting with an Uppercase Letter

**Objective:** Extract all tokens that begin with an uppercase letter from a fixed passage of *Alice's Adventures in Wonderland* [3].

**RegEx pattern:** `\b[A-Z][a-zA-Z]*\b`

| Component | Meaning |
|-----------|--------|
| `\b` | Word boundary anchor — ensures we match whole words, not sub-strings |
| `[A-Z]` | The first character must be an uppercase ASCII letter |
| `[a-zA-Z]*` | Zero or more subsequent letters (upper or lower) |
| `\b` | Closing word boundary |


In [ ]:
# ─────────────────────────────────────────────
# Task 2a – Uppercase Word Extraction
# ─────────────────────────────────────────────

text_2a = """Alice was beginning to get very tired of sitting by her sister on the bank,
and of having nothing to do. Once or twice she had peeped into the book
her sister was reading, but it had no pictures or conversations in it, "and
what is the use of a book," thought Alice, "without pictures or
conversations?"""

# \b[A-Z][a-zA-Z]*\b
#   \b        – word boundary (prevents matching mid-word)
#   [A-Z]     – exactly one uppercase letter as the first character
#   [a-zA-Z]* – zero or more subsequent letters (mixed case allowed)
#   \b        – closing word boundary
pattern_2a = r"\b[A-Z][a-zA-Z]*\b"

uppercase_words = re.findall(pattern_2a, text_2a)

print("=== Task 2a Output ===")
print(f"RegEx pattern  : {pattern_2a}")
print(f"Matches found  : {uppercase_words}")
print(f"Total count    : {len(uppercase_words)}")

### 2b. Corpus-Level Substitution: *Whale* → *leviathan* in *Moby-Dick*


In [ ]:
# ─────────────────────────────────────────────
# Task 2b – Whale → Leviathan Substitution
# ─────────────────────────────────────────────
import os

PDF_PATH = "melville-moby_dick.pdf"   
OUTPUT_TXT = "moby_dick_leviathan.txt"

# ── Step 1: Load full text ────────────────────────────────────────────────────
# Strategy: attempt to read a pre-extracted .txt first (fast); fall back to
# pdfplumber extraction from the PDF if the .txt is absent.
TXT_PATH = "melville-moby_dick.txt"  # pre-extracted plain-text (if available)

if os.path.exists(TXT_PATH):
    print(f"[INFO] Reading text from '{TXT_PATH}'...")
    try:
        with open(TXT_PATH, "r", encoding="utf-8") as f:
            moby_text = f.read()
    except UnicodeDecodeError:
        with open(TXT_PATH, "r", encoding="latin-1") as f:
            moby_text = f.read()
elif os.path.exists(PDF_PATH):
    print(f"[INFO] '{TXT_PATH}' not found. Extracting text from '{PDF_PATH}'...")
    moby_text = ""
    with pdfplumber.open(PDF_PATH) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                moby_text += page_text + "\n"
    print(f"[INFO] Extraction complete. Total characters: {len(moby_text):,}")
else:
    raise FileNotFoundError(
        f"Neither '{TXT_PATH}' nor '{PDF_PATH}' was found.\n"
        "Please place one of them in the same directory as this notebook."
    )

# ── Step 2: Count all occurrences ────────────────────────────────────────────
# \bwhales?\b  matches 'whale' and 'whales' as whole words (case-insensitive)
pattern_2b = r"\bwhales?\b"

all_matches = re.findall(pattern_2b, moby_text, flags=re.IGNORECASE)
print(f"\n=== Task 2b Output ===")
print(f"RegEx pattern              : {pattern_2b}")
print(f"Total matches found        : {len(all_matches)}")

# ── Step 3: Replace first 10 occurrences ─────────────────────────────────────
# re.sub with count=10 limits replacements to the first ten matches.
# Replacement is lowercase 'leviathan' regardless of original casing.
replaced_text = re.sub(pattern_2b, "leviathan", moby_text, count=10, flags=re.IGNORECASE)

# ── Step 4: Write output file ────────────────────────────────────────────────
with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write(replaced_text)

print(f"First 10 occurrences replaced with 'leviathan'.")
print(f"Output saved to            : {OUTPUT_TXT}")

# ── Step 5: Verify – show the first snippet containing 'leviathan' ───────────
leviathan_matches = [(m.start(), m.end()) for m in re.finditer(r"leviathan", replaced_text, re.IGNORECASE)]
print(f"\nVerification – first replacement in context:")
start = leviathan_matches[0][0]
print(f"  ...{replaced_text[max(0, start-40):start+50]}...")

### 2c. Extracting Jack Sparrow's Lines from *Pirates of the Caribbean*


In [ ]:
# ─────────────────────────────────────────────
# Task 2c – Jack Sparrow Lines from NLTK webtext
# ─────────────────────────────────────────────

# ── Step 1: Download NLTK webtext corpus (no-op if already present) ──────────
nltk.download("webtext", quiet=True)

# ── Step 2: Load pirates.txt ─────────────────────────────────────────────────
raw_text = webtext.raw("pirates.txt")
lines = raw_text.splitlines()

# ── Step 3: Apply RegEx filter ───────────────────────────────────────────────
# Pattern breakdown:
#   ^\s*                       – optional leading whitespace
#   (?:                        – non-capturing group (speaker alternatives)
#     JACK(?:\s+SPARROW)?      –   'JACK' optionally followed by ' SPARROW'
#     |                        –   OR
#     Jack(?:\s+Sparrow)?      –   'Jack' optionally followed by ' Sparrow'
#   )                          – end alternation group
#   \s*:\s*                    – colon with optional surrounding whitespace
#   .+                         – one or more dialogue characters
pattern_2c = r"^\s*(?:JACK(?:\s+SPARROW)?|Jack(?:\s+Sparrow)?)\s*:\s*.+"

jack_lines = [line for line in lines if re.match(pattern_2c, line)]

# ── Step 4: Display results ───────────────────────────────────────────────────
print("=== Task 2c Output ===")
print(f"RegEx pattern used        : {pattern_2c}")
print(f"Total Jack Sparrow lines  : {len(jack_lines)}")
print()
print("First 10 lines:")
print("-" * 70)
for line in jack_lines[:10]:
    print(line)
print("-" * 70)
print(f"  ... and {len(jack_lines) - 10} more lines.")